
# 🌎 BioGuard — FIAP Global Solution 2026
## Monitoramento Inteligente de Queimadas e Impacto na Fauna Brasileira

Este notebook utiliza:
- 🔥 Dados do INPE BDQueimadas
- 🐾 Base local da IUCN Red List
- 🗺️ Mapas interativos
- 📊 Dashboards analíticos
- 🤖 Clusterização de hotspots
- 🧠 Índice de vulnerabilidade ambiental

> Versão estável sem dependência da API online da IUCN.


In [1]:

# ============================================================
# 📦 IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import folium
import plotly.express as px
import matplotlib.pyplot as plt
import tabulate

from folium.plugins import HeatMap
from sklearn.cluster import KMeans
from IPython.display import display, Markdown

pd.set_option("display.max_columns", None)



# 📂 Carregamento dos dados do INPE


In [2]:

# ============================================================
# 🔥 CARREGAR DADOS DO INPE
# ============================================================

arquivo = r"C:\Users\eferro\Downloads\bdqueimadas_2026-05-27_2026-05-28.csv"

df = pd.read_csv(arquivo)

# Padronizar colunas
df.columns = df.columns.str.lower().str.strip()

print("📋 Colunas disponíveis:\n")
print(df.columns.tolist())

df.head()


📋 Colunas disponíveis:

['datahora', 'satelite', 'pais', 'estado', 'municipio', 'bioma', 'diasemchuva', 'precipitacao', 'riscofogo', 'frp', 'latitude', 'longitude']


,datahora,satelite,pais,estado,municipio,bioma,diasemchuva,precipitacao,riscofogo,frp,latitude,longitude
0,2026/05/27 18:47:00,AQUA_M-T,Brasil,MINAS GERAIS,MONTE CARMELO,Cerrado,4,0.00,1.00,22.9,-18.88687,-47.29673
1,2026/05/27 18:47:00,AQUA_M-T,Brasil,DISTRITO FEDERAL,BRASÍLIA,Cerrado,10,0.00,0.97,21.9,-15.70154,-47.98667
2,2026/05/27 18:47:00,AQUA_M-T,Brasil,MINAS GERAIS,JACUTINGA,Mata Atlântica,2,0.19,0.44,3.6,-22.34377,-46.62320
3,2026/05/27 18:47:00,AQUA_M-T,Brasil,MINAS GERAIS,MONTE CARMELO,Cerrado,4,0.00,1.00,5.4,-18.89564,-47.29528
4,2026/05/27 18:47:00,AQUA_M-T,Brasil,GOIÁS,LUZIÂNIA,Cerrado,4,0.00,0.99,15.9,-16.37163,-48.14189



# 🧹 Limpeza e preparação dos dados


In [3]:

# ============================================================
# 🧹 LIMPEZA
# ============================================================

rename_map = {}

if "latitude" in df.columns:
    rename_map["latitude"] = "lat"

if "longitude" in df.columns:
    rename_map["longitude"] = "lon"

df = df.rename(columns=rename_map)

# Detectar coluna de bioma
coluna_bioma = None

for col in df.columns:
    if "bioma" in col:
        coluna_bioma = col
        break

print(f"🌱 Coluna de bioma encontrada: {coluna_bioma}")

# Filtrar Amazônia e Pantanal
if coluna_bioma:

    biomas = ["Pantanal", "Amazônia", "Amazonia"]

    df = df[df[coluna_bioma].isin(biomas)]

print(f"🔥 Registros após filtro: {len(df)}")

# Remover nulos
df = df.dropna(subset=["lat", "lon"])

df.head()


🌱 Coluna de bioma encontrada: bioma
🔥 Registros após filtro: 95


,datahora,satelite,pais,estado,municipio,bioma,diasemchuva,precipitacao,riscofogo,frp,lat,lon
37,2026/05/27 18:49:00,AQUA_M-T,Brasil,PARÁ,ÁGUA AZUL DO NORTE,Amazônia,11,0.0,0.66,11.2,-6.99625,-50.47430
38,2026/05/27 18:49:00,AQUA_M-T,Brasil,MATO GROSSO,PARANATINGA,Amazônia,10,0.3,0.36,40.1,-13.38040,-54.26106
40,2026/05/27 18:49:00,AQUA_M-T,Brasil,PARÁ,ALTAMIRA,Amazônia,7,0.0,0.87,9.8,-6.53285,-55.06866
45,2026/05/27 18:49:00,AQUA_M-T,Brasil,PARÁ,CUMARU DO NORTE,Amazônia,9,0.0,0.56,12.2,-7.68845,-51.11501
49,2026/05/27 18:49:00,AQUA_M-T,Brasil,PARÁ,CANAÃ DOS CARAJÁS,Amazônia,12,0.0,1.00,6.8,-6.42609,-50.18463



# 🐾 Base local de espécies ameaçadas


In [4]:

# ============================================================
# 🐾 BASE LOCAL IUCN
# ============================================================

dados_especies = [
    {
        "especie": "Panthera onca",
        "nome_comum": "Onça-pintada",
        "bioma": "Pantanal",
        "categoria_iucn": "NT"
    },
    {
        "especie": "Anodorhynchus hyacinthinus",
        "nome_comum": "Arara-azul",
        "bioma": "Pantanal",
        "categoria_iucn": "VU"
    },
    {
        "especie": "Myrmecophaga tridactyla",
        "nome_comum": "Tamanduá-bandeira",
        "bioma": "Pantanal",
        "categoria_iucn": "VU"
    },
    {
        "especie": "Chrysocyon brachyurus",
        "nome_comum": "Lobo-guará",
        "bioma": "Pantanal",
        "categoria_iucn": "NT"
    },
    {
        "especie": "Saguinus bicolor",
        "nome_comum": "Sauim-de-coleira",
        "bioma": "Amazônia",
        "categoria_iucn": "CR"
    },
    {
        "especie": "Ateles paniscus",
        "nome_comum": "Macaco-aranha",
        "bioma": "Amazônia",
        "categoria_iucn": "VU"
    },
    {
        "especie": "Cacajao calvus",
        "nome_comum": "Uacari-branco",
        "bioma": "Amazônia",
        "categoria_iucn": "VU"
    }
]

df_especies = pd.DataFrame(dados_especies)

df_especies


,especie,nome_comum,bioma,categoria_iucn
0,Panthera onca,Onça-pintada,Pantanal,NT
1,Anodorhynchus hyacinthinus,Arara-azul,Pantanal,VU
2,Myrmecophaga tridactyla,Tamanduá-bandeira,Pantanal,VU
3,Chrysocyon brachyurus,Lobo-guará,Pantanal,NT
4,Saguinus bicolor,Sauim-de-coleira,Amazônia,CR
5,Ateles paniscus,Macaco-aranha,Amazônia,VU
6,Cacajao calvus,Uacari-branco,Amazônia,VU



# ⚖️ Índice de risco ambiental


In [5]:

# ============================================================
# ⚖️ PESOS IUCN
# ============================================================

PESO_IUCN = {
    "CR": 5,
    "EN": 4,
    "VU": 3,
    "NT": 2,
    "LC": 1
}

df_especies["peso_risco"] = df_especies["categoria_iucn"].map(PESO_IUCN)

df_especies


,especie,nome_comum,bioma,categoria_iucn,peso_risco
0,Panthera onca,Onça-pintada,Pantanal,NT,2
1,Anodorhynchus hyacinthinus,Arara-azul,Pantanal,VU,3
2,Myrmecophaga tridactyla,Tamanduá-bandeira,Pantanal,VU,3
3,Chrysocyon brachyurus,Lobo-guará,Pantanal,NT,2
4,Saguinus bicolor,Sauim-de-coleira,Amazônia,CR,5
5,Ateles paniscus,Macaco-aranha,Amazônia,VU,3
6,Cacajao calvus,Uacari-branco,Amazônia,VU,3


In [6]:

# ============================================================
# 🧠 CÁLCULO DE RISCO
# ============================================================

qtd_queimadas = len(df)

media_risco = df_especies["peso_risco"].mean()

indice_vulnerabilidade = qtd_queimadas * media_risco

print(f"🔥 Queimadas registradas: {qtd_queimadas}")
print(f"🐾 Média de risco das espécies: {round(media_risco,2)}")
print(f"🚨 Índice de vulnerabilidade: {round(indice_vulnerabilidade,2)}")

if indice_vulnerabilidade < 500:
    nivel = "Baixo"

elif indice_vulnerabilidade < 2000:
    nivel = "Moderado"

elif indice_vulnerabilidade < 5000:
    nivel = "Alto"

else:
    nivel = "Crítico"

print(f"⚠️ Classificação ambiental: {nivel}")


🔥 Queimadas registradas: 95
🐾 Média de risco das espécies: 3.0
🚨 Índice de vulnerabilidade: 285.0
⚠️ Classificação ambiental: Baixo



# 🗺️ Mapa interativo de queimadas


In [7]:
# ============================================================
# 🗺️ MAPA INTERATIVO
# ============================================================

# Detectar colunas automaticamente
col_lat = None
col_lon = None

for col in df.columns:

    if "lat" in col.lower():
        col_lat = col

    if "lon" in col.lower() or "long" in col.lower():
        col_lon = col

print("Latitude:", col_lat)
print("Longitude:", col_lon)

# Verificar se encontrou
if col_lat is None or col_lon is None:

    print("❌ Colunas de coordenadas não encontradas!")

else:

    # Remover nulos
    mapa_df = df.dropna(subset=[col_lat, col_lon])

    print("🔥 Registros usados no mapa:", len(mapa_df))

    # Criar mapa
    mapa = folium.Map(
        location=[-14, -55],
        zoom_start=4,
        tiles="cartodbpositron"
    )

    # Heatmap
    heat_data = mapa_df[[col_lat, col_lon]].values.tolist()

    HeatMap(
        heat_data,
        radius=15
    ).add_to(mapa)

    # Espécies
    marcadores = [
        [-16, -56, "🐾 Onça-pintada"],
        [-18, -57, "🐦 Arara-azul"],
        [-3, -60, "🐒 Sauim-de-coleira"],
        [-5, -63, "🐒 Macaco-aranha"]
    ]

    for lat, lon, nome in marcadores:

        folium.Marker(
            location=[lat, lon],
            popup=nome,
            icon=folium.Icon(color="green")
        ).add_to(mapa)

    mapa


Latitude: lat
Longitude: lon
🔥 Registros usados no mapa: 95



# 📊 Dashboard — Queimadas por bioma


In [8]:

# ============================================================
# 📊 DASHBOARD BIOMAS
# ============================================================

if coluna_bioma:

    fig = px.histogram(
        df,
        x=coluna_bioma,
        color=coluna_bioma,
        title="🔥 Queimadas por Bioma"
    )

    fig.show()



# 🐾 Dashboard — Espécies monitoradas


In [9]:

# ============================================================
# 🐾 DASHBOARD IUCN
# ============================================================

fig = px.pie(
    df_especies,
    names="categoria_iucn",
    title="🐾 Distribuição das Categorias IUCN"
)

fig.show()



# 🤖 Clusterização de hotspots ambientais


In [10]:

# ============================================================
# 🤖 HOTSPOTS
# ============================================================

dados_cluster = df[["lat", "lon"]]

if len(dados_cluster) >= 5:

    modelo = KMeans(
        n_clusters=5,
        random_state=42,
        n_init=10
    )

    modelo.fit(dados_cluster)

    dados_cluster["cluster"] = modelo.labels_

    fig = px.scatter_mapbox(
        dados_cluster,
        lat="lat",
        lon="lon",
        color="cluster",
        zoom=3,
        title="🔥 Hotspots Ambientais",
        mapbox_style="carto-positron"
    )

    fig.show()

else:
    print("❌ Poucos dados para clusterização")


C:\Users\eferro\AppData\Local\Temp\ipykernel_38652\2908731552.py:19: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/




# 📄 Relatório ambiental automático


In [11]:

# ============================================================
# 📄 RELATÓRIO
# ============================================================

display(Markdown(f'''
# 🌎 Relatório Ambiental — BioGuard

## 🔥 Queimadas monitoradas
**{qtd_queimadas}** focos detectados.

## ⚠️ Nível de risco ambiental
**{nivel}**

## 🐾 Espécies monitoradas
{", ".join(df_especies["nome_comum"].tolist())}

## 📊 Categorias IUCN
{df_especies[["nome_comum", "categoria_iucn"]].to_markdown(index=False)}

---

## 🌱 Conclusão

O BioGuard utiliza dados espaciais do INPE para identificar impactos ambientais na fauna brasileira, conectando queimadas e espécies ameaçadas da Amazônia e Pantanal.

O sistema permite visualizar regiões críticas, analisar riscos ambientais e auxiliar estratégias de preservação da biodiversidade.
'''))



# 🌎 Relatório Ambiental — BioGuard

## 🔥 Queimadas monitoradas
**95** focos detectados.

## ⚠️ Nível de risco ambiental
**Baixo**

## 🐾 Espécies monitoradas
Onça-pintada, Arara-azul, Tamanduá-bandeira, Lobo-guará, Sauim-de-coleira, Macaco-aranha, Uacari-branco

## 📊 Categorias IUCN
| nome_comum        | categoria_iucn   |
|:------------------|:-----------------|
| Onça-pintada      | NT               |
| Arara-azul        | VU               |
| Tamanduá-bandeira | VU               |
| Lobo-guará        | NT               |
| Sauim-de-coleira  | CR               |
| Macaco-aranha     | VU               |
| Uacari-branco     | VU               |

---

## 🌱 Conclusão

O BioGuard utiliza dados espaciais do INPE para identificar impactos ambientais na fauna brasileira, conectando queimadas e espécies ameaçadas da Amazônia e Pantanal.

O sistema permite visualizar regiões críticas, analisar riscos ambientais e auxiliar estratégias de preservação da biodiversidade.



# 💾 Exportação dos resultados


In [12]:

# ============================================================
# 💾 EXPORTAR RESULTADOS
# ============================================================

df.to_csv("dados_filtrados_bioguard.csv", index=False)

df_especies.to_csv("especies_monitoradas.csv", index=False)

print("✅ Arquivos exportados com sucesso!")


✅ Arquivos exportados com sucesso!
